# Part A - EDA and preprocessing for credit_applicants.csv

In [1]:
## Load necessary libraries

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
RANDOM_STATE = 42
NUMERIC_FEATURES = [
    "age",
    "monthly_income_inr",
    "existing_loans_count",
    "credit_utilization_ratio",
    "upi_monthly_inflow_inr",
    "bounced_payments_count",
    "credit_bureau_score",
]

report = {}

In [ ]:
## Step 1. Load data, report default rate + % missing credit_bureau_score.

In [5]:
def load_and_flag(path="/content/credit_applicants.csv"):
    df = pd.read_csv(path)

    default_rate = df["default"].mean()
    pct_missing_bureau = df["credit_bureau_score"].isna().mean() * 100

    report["n_rows"] = len(df)
    report["measured_default_rate"] = round(float(default_rate), 4)
    report["pct_missing_bureau_score"] = round(float(pct_missing_bureau), 2)
    report["n_missing_bureau_score"] = int(df["credit_bureau_score"].isna().sum())
    # Step 2: thin-file flag, computed straight from raw missingness.
    df["is_thin_file"] = df["credit_bureau_score"].isna().astype(int)

    return df

In [6]:
def split_impute_encode_scale(df):
    X = df.drop(columns=["default", "applicant_id"])
    y = df["default"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )
    train_median = X_train["credit_bureau_score"].median()
    report["train_bureau_score_median_used_for_imputation"] = float(train_median)

    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train["credit_bureau_score"] = X_train["credit_bureau_score"].fillna(train_median)
    X_test["credit_bureau_score"] = X_test["credit_bureau_score"].fillna(train_median)
    X_train = pd.get_dummies(X_train, columns=["employment_type"], drop_first=True)
    X_test = pd.get_dummies(X_test, columns=["employment_type"], drop_first=True)
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
    scaler = StandardScaler()
    X_train[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
    X_test[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

    return X_train, X_test, y_train, y_test, scaler

In [8]:
if __name__ == "__main__":
    import os
    os.makedirs('outputs', exist_ok=True)
    df = load_and_flag()
    X_train, X_test, y_train, y_test, scaler = split_impute_encode_scale(df)

    report["train_rows"] = len(X_train)
    report["test_rows"] = len(X_test)
    report["train_default_rate"] = round(float(y_train.mean()), 4)
    report["test_default_rate"] = round(float(y_test.mean()), 4)
    report["feature_columns"] = list(X_train.columns)

    print(json.dumps(report, indent=2))

    X_train.to_csv("outputs/X_train.csv", index=False)
    X_test.to_csv("outputs/X_test.csv", index=False)
    y_train.to_csv("outputs/y_train.csv", index=False)
    y_test.to_csv("outputs/y_test.csv", index=False)
    with open("outputs/part_a_report.json", "w") as f:
        json.dump(report, f, indent=2)

{
  "n_rows": 400,
  "measured_default_rate": 0.2025,
  "pct_missing_bureau_score": 20.0,
  "n_missing_bureau_score": 80,
  "train_bureau_score_median_used_for_imputation": 612.0,
  "train_rows": 300,
  "test_rows": 100,
  "train_default_rate": 0.2033,
  "test_default_rate": 0.2,
  "feature_columns": [
    "age",
    "monthly_income_inr",
    "existing_loans_count",
    "credit_utilization_ratio",
    "upi_monthly_inflow_inr",
    "bounced_payments_count",
    "credit_bureau_score",
    "is_thin_file",
    "employment_type_salaried",
    "employment_type_self_employed"
  ]
}
